# SentencePiece でサブワードモデルを学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/05_subword/sentencepiece_train.ipynb)

NLP クックブック 05-01 ／ サブワードとトークナイザ

自分のコーパスからサブワード分割モデルを作りたい。日本語でも事前の単語分割なしで学習できる。

In [1]:
!pip install -q sentencepiece

In [2]:
import io, re, urllib.request, zipfile

# 学習用コーパス: 青空文庫「吾輩は猫である」
url = "https://www.aozora.gr.jp/cards/000148/files/789_ruby_5639.zip"
with urllib.request.urlopen(url) as r:
    z = zipfile.ZipFile(io.BytesIO(r.read()))
text = z.read(z.namelist()[0]).decode("cp932")
text = re.split(r"-{50,}", text)[2].split("底本：")[0]          # ヘッダと奥付を除く
text = re.sub(r"《.+?》|［＃.+?］|｜", "", text)                 # ルビ・注記を除く
lines = [s for s in re.split(r"(?<=。)|\r\n", text) if s and s.strip()]
with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(s.strip() for s in lines))
print(len(lines), "文")

9069 文


In [3]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input="corpus.txt", model_prefix="neko", vocab_size=4000,
    model_type="unigram",        # "bpe" も選べる
    character_coverage=0.9995,   # 日本語・中国語は 0.9995、英語などは 1.0
    minloglevel=1,
)
sp = spm.SentencePieceProcessor(model_file="neko.model")
s = "吾輩は猫である。名前はまだ無い。"
print(sp.encode(s, out_type=str))
print(sp.encode(s))
print(sp.decode(sp.encode(s)))

['▁吾輩は', '猫', 'である', '。', '名', '前', 'は', 'まだ', '無', 'い', '。']
[191, 88, 27, 3, 125, 152, 10, 203, 131, 19, 3]
吾輩は猫である。名前はまだ無い。


- `▁` (U+2581) は空白（単語境界）を表す。デコードすると元の文に完全に戻る（可逆）のが SentencePiece の特徴。
- 語彙サイズは、翻訳なら 8k〜32k、小さなコーパスなら 4k〜8k が目安。